# Starck, Fadili & Murtagh (2007) — UWT / IUWT Implementation

**Paper**: Starck, J.-L., Fadili, J., & Murtagh, F. (2007). *The Undecimated Wavelet Decomposition and its Reconstruction.* IEEE Trans. Image Process., **16**(2), 297–309.

This notebook reproduces the paper's key algorithms from scratch:
1. The **à-trous (with-holes) algorithm** with cubic-spline filter $h^{1D}=[1,4,6,4,1]/16$.
2. The **Isotropic Undecimated Wavelet Transform (IUWT)** as a difference of resolutions ($w_{j+1} = c_j - c_{j+1}$).
3. The trivial **direct reconstruction** $c_0 = c_J + \sum_j w_j$ (Eq. 10).
4. **Soft-thresholding denoising** on the IUWT detail bands.
5. The **Landweber/POCS iterative reconstruction** with positivity (Eq. 25).
6. A **comparison experiment** (direct vs iterative reconstruction) on a noisy synthetic image.

이 노트북은 논문의 핵심 — IUWT 의 분해/재구성, soft-thresholding 디노이징, Landweber 반복 — 을 처음부터 구현하고 동기 실험으로 검증한다.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d
from scipy.ndimage import gaussian_filter

try:
    from skimage import data
    HAVE_SKIMAGE = True
except ImportError:
    HAVE_SKIMAGE = False

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Part 1: à-trous filter and dilated convolution / à-trous 필터와 dilated convolution

The à-trous algorithm replaces every downsampling step with **inserting $2^j-1$ zeros** between successive taps of the analysis filter, so the kernel acts on coarser-scale pixel spacings while keeping the image size constant. We implement the dilated kernel and a separable-2D convolution helper.

In [ ]:
def cubic_bspline_filter() -> np.ndarray:
    """Return the 1-D cubic B-spline filter (Astro filter) from Eq. 6.

    Returns:
        np.ndarray: 1-D filter of length 5, h^{1D} = [1, 4, 6, 4, 1] / 16.
    """
    return np.array([1.0, 4.0, 6.0, 4.0, 1.0]) / 16.0


def dilate_filter(h: np.ndarray, j: int) -> np.ndarray:
    """Insert (2**j - 1) zeros between successive taps of h.

    Implements the à-trous (with-holes) construction h^{(j)} (Eq. 1).

    Args:
        h: 1-D analysis filter.
        j: Scale level (0-indexed); j=0 returns h unchanged.

    Returns:
        Dilated 1-D filter.
    """
    if j == 0:
        return h.copy()
    step = 2 ** j
    n = len(h)
    out = np.zeros((n - 1) * step + 1)
    out[::step] = h
    return out


def separable_conv2d(image: np.ndarray, h1d: np.ndarray) -> np.ndarray:
    """Apply a separable 2-D filter to an image with mirror boundary.

    Args:
        image: 2-D float array.
        h1d: 1-D filter; the 2-D filter is h1d (rows) ⊗ h1d (cols).

    Returns:
        Filtered image of the same shape.
    """
    row = h1d.reshape(1, -1)
    col = h1d.reshape(-1, 1)
    tmp = convolve2d(image, row, mode='same', boundary='symm')
    return convolve2d(tmp, col, mode='same', boundary='symm')


h_base = cubic_bspline_filter()
print('h^{1D} =', h_base)
print('Sum (should be 1) =', h_base.sum())
for j in range(3):
    print(f'h^({j}) =', dilate_filter(h_base, j))

## Part 2: IUWT analysis and synthesis / IUWT 분해와 합성

Following Eq. (12) of the paper, the IUWT defines a single isotropic detail band per scale via

$$
c_{j+1} = h^{(j)} * c_j, \qquad w_{j+1} = c_j - c_{j+1}.
$$

Synthesis is the trivial sum $c_0 = c_J + \sum_{j=1}^{J} w_j$ (Eq. 10).

In [ ]:
def iuwt_decompose(image: np.ndarray, n_scales: int) -> tuple[list[np.ndarray], np.ndarray]:
    """Forward IUWT: produce n_scales detail bands and a coarse residual.

    Implements w_{j+1} = c_j - c_{j+1} where c_{j+1} = h^{(j)} * c_j.

    Args:
        image: 2-D float input image.
        n_scales: Number of detail bands to compute.

    Returns:
        details: List of n_scales detail images, same shape as input.
        coarse: The smoothest residual c_J.
    """
    h = cubic_bspline_filter()
    c_prev = image.astype(np.float64)
    details: list[np.ndarray] = []
    for j in range(n_scales):
        h_j = dilate_filter(h, j)
        c_next = separable_conv2d(c_prev, h_j)
        details.append(c_prev - c_next)
        c_prev = c_next
    return details, c_prev


def iuwt_reconstruct(details: list[np.ndarray], coarse: np.ndarray) -> np.ndarray:
    """Direct IUWT synthesis: c_0 = c_J + sum_{j} w_j (Eq. 10).

    Args:
        details: List of detail bands from iuwt_decompose.
        coarse: Coarse residual c_J.

    Returns:
        Reconstructed image.
    """
    out = coarse.copy()
    for w in details:
        out = out + w
    return out


# Round-trip test on a synthetic image / 합성 테스트로 round-trip 검증.
test_img = np.zeros((128, 128))
test_img[40:90, 40:90] = 1.0
test_img += 0.4 * np.exp(-((np.arange(128)[None, :] - 70) ** 2 + (np.arange(128)[:, None] - 60) ** 2) / 200)

details, coarse = iuwt_decompose(test_img, n_scales=4)
recon = iuwt_reconstruct(details, coarse)
print('Round-trip max abs error:', np.max(np.abs(test_img - recon)))

## Part 3: Visualise the IUWT decomposition / IUWT 분해 시각화

Show the four detail bands and the coarse residual side by side. Each detail band $w_j$ contains structure of approximate scale $2^j$ pixels — this is the multiresolution that subsequent solar-coronal enhancement (NRGF, MGN) re-uses.

In [ ]:
fig, axes = plt.subplots(1, len(details) + 2, figsize=(14, 3.0))
axes[0].imshow(test_img, cmap='gray'); axes[0].set_title('input'); axes[0].axis('off')
for j, w in enumerate(details):
    vlim = max(abs(w.min()), abs(w.max()))
    axes[j + 1].imshow(w, cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    axes[j + 1].set_title(f'w_{j+1}')
    axes[j + 1].axis('off')
axes[-1].imshow(coarse, cmap='gray'); axes[-1].set_title(f'c_{len(details)}'); axes[-1].axis('off')
plt.tight_layout()
plt.show()

## Part 4: Soft-threshold denoising / Soft-threshold 디노이징

Add Gaussian noise, decompose, soft-threshold each detail band at $T_j = K\sigma_j$ where $\sigma_j$ is estimated from the median-absolute-deviation (MAD) of the band, and reconstruct directly.

In [ ]:
def soft_threshold(x: np.ndarray, t: float) -> np.ndarray:
    """Element-wise soft thresholding: sgn(x) * max(|x| - t, 0)."""
    return np.sign(x) * np.maximum(np.abs(x) - t, 0.0)


def estimate_noise_mad(band: np.ndarray) -> float:
    """Estimate Gaussian noise sigma via the MAD rule on a wavelet detail band.

    Args:
        band: 2-D detail-band array.

    Returns:
        Estimated standard deviation, sigma = MAD / 0.6745.
    """
    return float(np.median(np.abs(band)) / 0.6745)


def iuwt_denoise_direct(image: np.ndarray, n_scales: int = 4, k: float = 3.0) -> tuple[np.ndarray, list[np.ndarray]]:
    """Direct denoising: forward IUWT, soft-threshold details, direct synthesis.

    Args:
        image: Noisy 2-D image.
        n_scales: Number of detail bands.
        k: Threshold multiplier; T_j = k * sigma_j (sigma_j from finest band rescaled).

    Returns:
        Denoised image and the kept (thresholded) detail bands.
    """
    details, coarse = iuwt_decompose(image, n_scales)
    sigma = estimate_noise_mad(details[0])
    # Cubic-spline filter sigma propagation factors (Starck 2002 Table 2.1, p. 31). / Sigma 전파계수.
    sigma_factors = [0.890, 0.201, 0.086, 0.041, 0.020, 0.010][:n_scales]
    kept = []
    for j, w in enumerate(details):
        t = k * sigma * sigma_factors[j] / sigma_factors[0]
        kept.append(soft_threshold(w, t))
    return iuwt_reconstruct(kept, coarse), kept


# Synthetic test image with noise / 잡음을 더한 테스트 영상 생성
if HAVE_SKIMAGE:
    clean = data.camera().astype(np.float64) / 255.0
    clean = clean[::2, ::2]  # 256x256 -> faster
else:
    yy, xx = np.mgrid[0:256, 0:256]
    clean = 0.5 * (np.sin(yy * 0.05) * np.cos(xx * 0.07) + 1.0)
    clean[80:160, 80:160] += 0.3
    clean = np.clip(clean, 0, 1)

noise_sigma = 0.08
noisy = clean + noise_sigma * rng.standard_normal(clean.shape)
denoised_direct, kept_details = iuwt_denoise_direct(noisy, n_scales=4, k=3.0)
print(f'Input PSNR(noisy)    = {10*np.log10(1.0/np.mean((clean-noisy)**2)):.2f} dB')
print(f'Output PSNR(direct)  = {10*np.log10(1.0/np.mean((clean-denoised_direct)**2)):.2f} dB')

## Part 5: Landweber / POCS iterative reconstruction / Landweber 반복 재구성

Eq. (25) of the paper:

$$
\tilde S^{n+1} = P_+\!\left(\tilde S^{n} + \mathcal R\!\left[\alpha_T - \mathcal W \tilde S^{n}\right]\right).
$$

We treat the **multiresolution support** $M$ as the locations of coefficients that survived thresholding (sign-preserving) and force the iterate to keep the thresholded values where $M=1$, while allowing it to be re-estimated where $M=0$. For the IUWT, $\mathcal R = \sum$ (just sum the bands) and $\mathcal W$ = `iuwt_decompose`.

In [ ]:
def iuwt_denoise_iterative(
    image: np.ndarray,
    n_scales: int = 4,
    k: float = 3.0,
    n_iter: int = 25,
    enforce_positivity: bool = True,
) -> np.ndarray:
    """Landweber/POCS iterative reconstruction (Eq. 25).

    Args:
        image: Noisy 2-D image.
        n_scales: Number of IUWT detail bands.
        k: Threshold multiplier as in iuwt_denoise_direct.
        n_iter: Number of Landweber iterations.
        enforce_positivity: Apply pixel-wise positivity projection P_+.

    Returns:
        Denoised image after n_iter iterations.
    """
    # 1) build target thresholded coefficients alpha_T and support mask M / 임계처리된 alpha_T 와 support 계산
    details, coarse = iuwt_decompose(image, n_scales)
    sigma = estimate_noise_mad(details[0])
    sigma_factors = [0.890, 0.201, 0.086, 0.041, 0.020, 0.010][:n_scales]
    alpha_T = []
    masks = []
    for j, w in enumerate(details):
        t = k * sigma * sigma_factors[j] / sigma_factors[0]
        thr = soft_threshold(w, t)
        alpha_T.append(thr)
        masks.append(np.abs(thr) > 0)

    # 2) iterate / 반복
    s = iuwt_reconstruct(alpha_T, coarse)
    for _ in range(n_iter):
        d_curr, c_curr = iuwt_decompose(s, n_scales)
        # residual in coefficient space, restricted to support / support 영역에서 잔차
        residual_details = [
            np.where(M, alpha_T[j] - d_curr[j], 0.0)
            for j, M in enumerate(masks)
        ]
        s = s + iuwt_reconstruct(residual_details, np.zeros_like(c_curr))
        if enforce_positivity:
            s = np.maximum(s, 0.0)
    return s


denoised_iter = iuwt_denoise_iterative(noisy, n_scales=4, k=3.0, n_iter=25)
print(f'Output PSNR(iter)    = {10*np.log10(1.0/np.mean((clean-denoised_iter)**2)):.2f} dB')

## Part 6: Visual comparison / 시각적 비교

Compare the noisy input, direct denoising and iterative denoising side by side. Iterative reconstruction tends to recover edges with less ringing while preserving overall photometry — exactly the effect Fig. 9 of the paper illustrates.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, ttl in zip(
    axes,
    [clean, noisy, denoised_direct, denoised_iter],
    ['clean', f'noisy (σ={noise_sigma})', 'direct IUWT', 'iter. IUWT (Eq. 25)'],
):
    ax.imshow(np.clip(img, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(ttl)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Cross-section through a row to expose ringing / ringing 노출용 한 행 단면
row = clean.shape[0] // 2
plt.figure(figsize=(11, 3.5))
plt.plot(clean[row], 'k-', label='clean', lw=1.4)
plt.plot(noisy[row], color='0.6', alpha=0.5, label='noisy', lw=0.8)
plt.plot(denoised_direct[row], 'C0', label='direct IUWT', lw=1.2)
plt.plot(denoised_iter[row], 'C3', label='iterative IUWT', lw=1.2)
plt.title('Row cross-section — ringing comparison'); plt.legend(); plt.tight_layout(); plt.show()

## Part 7: Multi-K threshold experiment / 임계값 K 변화 실험

Sweep the threshold multiplier $K\in\{2,3,4,5,6\}$ and compare direct vs iterative reconstruction in PSNR. This reproduces the qualitative shape of the paper's Fig. 6 / Fig. 8 nonlinear-approximation curves.

In [ ]:
def psnr(reference: np.ndarray, candidate: np.ndarray, peak: float = 1.0) -> float:
    """Peak Signal-to-Noise Ratio in dB."""
    mse = float(np.mean((reference - candidate) ** 2))
    if mse <= 0:
        return float('inf')
    return 10.0 * np.log10(peak ** 2 / mse)


ks = [2.0, 3.0, 4.0, 5.0, 6.0]
psnrs_direct = []
psnrs_iter = []
for k in ks:
    di, _ = iuwt_denoise_direct(noisy, n_scales=4, k=k)
    it = iuwt_denoise_iterative(noisy, n_scales=4, k=k, n_iter=20)
    psnrs_direct.append(psnr(clean, di))
    psnrs_iter.append(psnr(clean, it))

plt.figure(figsize=(7, 4))
plt.plot(ks, psnrs_direct, 'o-', label='direct')
plt.plot(ks, psnrs_iter, 's-', label='iterative (Eq. 25)')
plt.xlabel('threshold multiplier K (T = K·σ_j)')
plt.ylabel('PSNR (dB)')
plt.title('Direct vs iterative IUWT denoising')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

for k, pd, pi in zip(ks, psnrs_direct, psnrs_iter):
    print(f'K={k:.1f}: direct={pd:.2f} dB, iterative={pi:.2f} dB, gain={pi-pd:+.2f} dB')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Multiscale image decomposition / 다중스케일 분해 | IUWT with cubic-spline $h^{1D}=[1,4,6,4,1]/16$ | Laplacian / Gaussian pyramids; learned U-Net features |
| Translation invariance / 평행이동 불변성 | Skip the decimation step (à-trous) | Stride-1 convolutions; cyclic shifts in CNNs |
| Sparse coefficient selection / 희소 계수 선택 | Hard / soft thresholding with $T_j=K\sigma_j$ | $\ell_1$ regularisation; ReLU + L1 in deep nets |
| Inverse map for redundant frame / 잉여 frame 의 역사상 | Landweber / POCS iteration (Eq. 25) | LISTA, unrolled networks, plug-and-play priors |
| Ringing-free synthesis / Ringing 없는 합성 | Positive synthesis $\tilde g = \delta + h$ | Non-negative dictionary learning; positivity layers |
| Solar-coronal enhancement / 코로나 강조 | IUWT pipelines | NRGF (#35), MGN (#38), NAFE; deep-learning EUV enhancement |